**To-Do**
1. Make it schema generic, so that whenever the tool assigns a specific schema, it can be used here and then we can proceed forward with the properties.

In [155]:
import json
import re
from dataclasses import dataclass
from urllib.parse import urlparse
from __future__ import annotations


import requests

In [156]:
@dataclass
class RepoRef:
    provider: str  # github | gitlab
    owner: str
    repo: str


def parse_repo_url(repo_url: str) -> RepoRef:
    parsed = urlparse(repo_url)
    host = parsed.netloc.lower()
    path_parts = [p for p in parsed.path.strip('/').split('/') if p]

    if len(path_parts) < 2:
        raise ValueError(f'Invalid repository URL: {repo_url}')

    owner, repo = path_parts[0], path_parts[1]
    if repo.endswith('.git'):
        repo = repo[:-4]

    if 'github.com' in host:
        return RepoRef(provider='github', owner=owner, repo=repo)
    if 'gitlab.com' in host:
        return RepoRef(provider='gitlab', owner=owner, repo=repo)

    raise ValueError('Only GitHub and GitLab URLs are supported.')


def candidate_readme_urls(ref: RepoRef) -> list[str]:
    branches = ['main', 'master']
    filenames = ['README.md', 'README.MD', 'readme.md', 'README.rst', 'README.txt']

    urls: list[str] = []
    for branch in branches:
        for filename in filenames:
            if ref.provider == 'github':
                urls.append(f'https://raw.githubusercontent.com/{ref.owner}/{ref.repo}/{branch}/{filename}')
            else:
                urls.append(f'https://gitlab.com/{ref.owner}/{ref.repo}/-/raw/{branch}/{filename}')
    return urls


def fetch_readme(ref: RepoRef, timeout: int = 20) -> tuple[str, str]:
    headers = {'User-Agent': 'maSMP-ollama-readme-test/1.0'}

    for url in candidate_readme_urls(ref):
        try:
            resp = requests.get(url, headers=headers, timeout=timeout)
            if resp.status_code == 200 :
                return resp.text, url
        except Exception:
            continue

    raise RuntimeError('README not found on main/master with common README filenames.')

In [157]:
repo_url_preview = 'https://github.com/CERN/TIGRE'
ref_preview = parse_repo_url(repo_url_preview)
readme_text_preview, readme_url_preview = fetch_readme(ref_preview, timeout=20)
# readme_chunk_preview = truncate_for_context(readme_text_preview)

print(f'[README URL] {readme_url_preview}')
print(f'[README Length] {len(readme_text_preview)} chars')
print(f'[README Preview] {readme_text_preview}')

[README URL] https://raw.githubusercontent.com/CERN/TIGRE/master/README.md
[README Length] 14111 chars
[README Preview] [![Documentation Status](https://readthedocs.org/projects/tigre/badge/?version=latest)](https://tigre.readthedocs.io/en/latest/?badge=latest)
<!-- ALL-CONTRIBUTORS-BADGE:START - Do not remove or modify this section -->
[![All Contributors](https://img.shields.io/badge/all_contributors-13-orange.svg?style=flat-square)](#contributors-)
<!-- ALL-CONTRIBUTORS-BADGE:END -->


TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox

TIGRE is an open-source toolbox for fast and accurate 3D tomographic 
reconstruction for any geometry.  Its focus is on iterative algorithms 
for improved image quality that have all been optimized to run on GPUs 
(including multi-GPUs) for improved speed. It combines the higher level 
abstraction of MATLAB or Python with the performance of CUDA at a lower level in order to make 
it both fast and easy to use.

Read the most up to date arti

In [158]:
def split_with_metadata(md_text):
    lines = md_text.split("\n")
    chunks = []

    current_chunk = {"heading": None, "level": None, "content": []}

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # --- CASE 1: Markdown # headings ---
        match = re.match(r'^(#{1,6})\s+(.*)', line)
        if match:
            if current_chunk["content"]:
                chunks.append(current_chunk)

            current_chunk = {
                "heading": match.group(2),
                "level": len(match.group(1)),
                "content": []
            }
            i += 1
            continue

        # --- CASE 2: Underline-style headings (==== or ----) ---
        if i + 1 < len(lines):
            next_line = lines[i + 1].strip()

            if re.match(r'^=+$', next_line):
                if current_chunk["content"]:
                    chunks.append(current_chunk)

                current_chunk = {
                    "heading": line,
                    "level": 1,
                    "content": []
                }
                i += 2
                continue

            elif re.match(r'^-+$', next_line):
                if current_chunk["content"]:
                    chunks.append(current_chunk)

                current_chunk = {
                    "heading": line,
                    "level": 2,
                    "content": []
                }
                i += 2
                continue

        # --- Normal content ---
        current_chunk["content"].append(lines[i])
        i += 1

    if current_chunk["content"]:
        chunks.append(current_chunk)

    return chunks

In [159]:
sections = split_with_metadata(readme_text_preview)
sections = [s for s in sections if s["heading"] is not None]
print(len(sections))
print([s["heading"] for s in sections])

10
['TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox', 'TIGRE features', 'Installation', 'Getting started', 'FAQ', 'Gallery', 'Further Reading', 'Contact', 'Licensing', 'Contributors']


In [160]:
# Checking the length of sections and the longest content section for context size estimation

len(sections)
max(len("\n".join(s["content"])) for s in sections)


6802

In [161]:
def hybrid_chunking(section, max_chars=1200, overlap=200):
    text = "\n".join(section["content"]).strip()
    
    # If small enough → keep as is
    if len(text) <= max_chars:
        return [{
            "heading": section["heading"],
            "content": text
        }]
    
    # Otherwise → split into overlapping chunks
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + max_chars
        chunk_text = text[start:end]
        
        chunks.append({
            "heading": section["heading"],
            "content": chunk_text
        })
        
        start += max_chars - overlap
    
    return chunks

In [162]:
all_chunks = []

for section in sections:
    all_chunks.extend(hybrid_chunking(section))

In [163]:
print(len(all_chunks))
print(all_chunks[0]["heading"])
print(len(all_chunks[0]["content"]))

19
TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox
1200


In [164]:
KEYWORDS = ['Installation', 'Getting started', 'FAQ', 'Gallery', 'Further Reading', 'Contact', 'Licensing', 'Contributors']

filtered_chunks = [
    c for c in all_chunks
    if any(k in (c["heading"] or "") for k in KEYWORDS)
]

print(len(all_chunks))
print(f'all chunks: {[c["heading"] for c in all_chunks]}')
print(f'filtered chunks: {[c["heading"] for c in filtered_chunks]}')
print(len(filtered_chunks))

19
all chunks: ['TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox', 'TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox', 'TIGRE features', 'TIGRE features', 'Installation', 'Getting started', 'FAQ', 'Gallery', 'Further Reading', 'Contact', 'Licensing', 'Licensing', 'Contributors', 'Contributors', 'Contributors', 'Contributors', 'Contributors', 'Contributors', 'Contributors']
filtered chunks: ['Installation', 'Getting started', 'FAQ', 'Gallery', 'Further Reading', 'Contact', 'Licensing', 'Licensing', 'Contributors', 'Contributors', 'Contributors', 'Contributors', 'Contributors', 'Contributors', 'Contributors']
15


## Next step: property-aware retrieval (embeddings + keyword hybrid)

This block adds schema-generic retrieval before extraction.
If `sentence-transformers` is unavailable, it falls back to lexical-only ranking.

In [165]:
from typing import Any
import numpy as np

PROPERTY_QUERIES = {
    'license': ['license', 'licensing', 'copyright', 'spdx'],
    'installation': ['installation', 'install', 'pip', 'conda', 'requirements', 'setup'],
    'contact': ['contact', 'email', 'maintainer', 'author'],
    'contributors': [
        'contributors', 'contributor', 'authors', 'author', 'maintainers',
        'maintainer', 'team', 'credits', 'acknowledgements', 'thanks',
        'community', '@', 'github.com'
    ],
    'links': [
        'paper', 'publication', 'publications', 'cite', 'citation', 'arxiv',
        'doi', 'research', 'reference', 'further reading', 'docs', 'url', 'link', 'read the article'
    ],
    'description': [
        'overview', 'about', 'what is', 'purpose', 'project', 'repository',
        'toolkit', 'library', 'framework', 'package', 'goal', 'features',
        'introduction', 'summary', 'readme'
    ],
}

PROPERTY_SCHEMA_HINTS = {
    'license': 'normalized SPDX-style string if explicit, else null',
    'installation': 'short installation instruction summary, else null',
    'contact': 'email or contact link, else null',
    'contributors': 'list of contributors with name/github_url if available, else null',
    'links': 'list of relevant links as objects {title, url, is_working, status_code, relevance}, else null',
    'description': 'one or two concise lines describing what the repository does, else null',
}

def _safe_text(x: Any) -> str:
    return '' if x is None else str(x)

def prepare_chunk_records(chunks: list[dict]) -> list[dict]:
    records = []
    for i, c in enumerate(chunks):
        heading = _safe_text(c.get('heading')).strip()
        content = _safe_text(c.get('content')).strip()
        if not content:
            continue
        records.append({
            'chunk_id': i,
            'heading': heading,
            'content': content,
            'char_len': len(content),
            'full_text': f"Heading: {heading}\n\n{content}" if heading else content,
        })
    return records

chunk_records = prepare_chunk_records(all_chunks)
print(f'[Chunk Records] {len(chunk_records)} usable chunks')

[Chunk Records] 19 usable chunks


In [166]:
from sentence_transformers import SentenceTransformer

def build_retrieval_index(records: list[dict], model_name: str = 'sentence-transformers/all-MiniLM-L6-v2') -> dict:
    index = {
        'records': records,
        'embeddings': None,
        'model': None,
        'embedding_enabled': False,
        'model_name': model_name,
    }
    try:
        model = SentenceTransformer(model_name)
        vectors = model.encode([r['full_text'] for r in records], normalize_embeddings=True)
        index['embeddings'] = np.asarray(vectors, dtype=np.float32)
        index['model'] = model
        index['embedding_enabled'] = True
        print(f'[Embeddings] enabled ({model_name})')
    except Exception as exc:
        print(f'[Embeddings] disabled, lexical fallback only: {exc}')
    return index

def keyword_score(record: dict, query_terms: list[str]) -> float:
    heading = record['heading'].lower()
    content = record['content'].lower()
    score = 0.0
    for term in query_terms:
        t = term.lower()
        if t in heading:
            score += 2.0
        if t in content:
            score += 1.0
    return score

def retrieve_top_chunks(index: dict, property_name: str, top_k: int = 5, alpha: float = 0.75) -> list[dict]:
    terms = PROPERTY_QUERIES[property_name]
    records = index['records']

    kw = np.array([keyword_score(r, terms) for r in records], dtype=np.float32)
    if kw.max() > 0:
        kw = kw / kw.max()

    if index['embedding_enabled']:
        q = ' '.join(terms)
        qv = index['model'].encode([q], normalize_embeddings=True)[0]
        ev = index['embeddings']
        sem = ev @ np.asarray(qv, dtype=np.float32)
        sem = (sem + 1.0) / 2.0
        scores = alpha * sem + (1 - alpha) * kw
    else:
        scores = kw

    order = np.argsort(-scores)[:top_k]
    out = []
    for rank, idx in enumerate(order, start=1):
        r = dict(records[int(idx)])
        r['score'] = float(scores[int(idx)])
        r['rank'] = rank
        out.append(r)
    return out

retrieval_index = build_retrieval_index(chunk_records)

[Embeddings] enabled (sentence-transformers/all-MiniLM-L6-v2)


In [167]:
def check_ollama_ready(
    model: str,
    ollama_url: str = 'http://localhost:11435',
    timeout: int = 10,
 ) -> tuple[bool, str]:
    endpoint = ollama_url.rstrip('/') + '/api/tags'
    try:
        resp = requests.get(endpoint, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        return False, f'Cannot reach Ollama at {endpoint}: {exc}'

    if resp.status_code != 200:
        return False, f'Ollama responded with {resp.status_code}: {resp.text}'

    data = resp.json()
    model_names = [m.get('name', '') for m in data.get('models', [])]

    if model in model_names:
        return True, f'Ollama is ready. Model `{model}` found.'

    if not model_names:
        return False, f'Ollama is running but has no models. Run: ollama pull {model}'

    return False, f'Model `{model}` not found. Available: {model_names}'


def run_ollama(prompt: str, model: str, ollama_url: str = 'http://localhost:11435', timeout: int = 120) -> str:
    endpoint = ollama_url.rstrip('/') + '/api/generate'
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
        'format': 'json',
        'options': {
            'temperature': 0,
            'top_p': 0.7,
            'num_predict': 140,
        },
    }

    try:
        resp = requests.post(endpoint, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(
            f'Connection failed to {endpoint}. Ensure Ollama is running and model `{model}` is pulled.'
        ) from exc

    if resp.status_code != 200:
        raise RuntimeError(f'Ollama error {resp.status_code}: {resp.text}')

    data = resp.json()
    if 'response' not in data:
        raise RuntimeError(f'Unexpected Ollama response: {data}')

    return data['response']

In [168]:
model_for_check = 'qwen2.5:7b'
ok, msg = check_ollama_ready(model=model_for_check, ollama_url='http://localhost:11435', timeout=10)
print(msg)

Ollama is ready. Model `qwen2.5:7b` found.


In [169]:
def extract_json(text: str) -> dict:
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r'\{[\s\S]*\}', text)
    if not match:
        raise ValueError('Model output does not contain JSON.')

    return json.loads(match.group(0))


def extract_html(text: str) -> str:
    text = text.strip()

    # Case 1: full HTML payload
    if re.search(r'<\s*html\b', text, flags=re.IGNORECASE):
        return text

    # Case 2: capture common HTML blocks/snippets
    block_match = re.search(
        r'(<(table|ul|ol|div|section|article|p)[^>]*>[\s\S]*?</\2>)',
        text,
        flags=re.IGNORECASE,
    )
    if block_match:
        return block_match.group(1)

    # Case 3: fallback to first generic HTML tag pair
    generic_match = re.search(r'(<([a-zA-Z][a-zA-Z0-9]*)[^>]*>[\s\S]*?</\2>)', text)
    if generic_match:
        return generic_match.group(1)

    raise ValueError('Model output does not contain HTML.')



def extract_license_from_readme(readme_text: str) -> tuple[str | None, str | None]:
    patterns = [
        (r'bsd\s*[- ]?3\s*[- ]?clause|bsd-3-clause', 'BSD-3-Clause'),
        (r'bsd\s*[- ]?2\s*[- ]?clause|bsd-2-clause', 'BSD-2-Clause'),
        (r'\bmit\b(?:\s+license)?', 'MIT'),
        (r'apache\s*(license\s*)?2\.0|apache-2\.0', 'Apache-2.0'),
        (r'gpl\s*v?3|gnu\s+general\s+public\s+license\s*v?3', 'GPL-3.0'),
        (r'gpl\s*v?2|gnu\s+general\s+public\s+license\s*v?2', 'GPL-2.0'),
        (r'lgpl\s*v?3|lesser\s+general\s+public\s+license\s*v?3', 'LGPL-3.0'),
        (r'mpl\s*2\.0|mozilla\s+public\s+license\s*2\.0', 'MPL-2.0'),
    ]

    low = readme_text.lower()
    for pat, spdx in patterns:
        m = re.search(pat, low, flags=re.IGNORECASE)
        if m:
            start = max(0, m.start() - 80)
            end = min(len(readme_text), m.end() + 80)
            evidence = readme_text[start:end].replace('\n', ' ').strip()
            return spdx, evidence

    return None, None



In [170]:
def extract_license_from_readme(readme_text: str) -> tuple[str | None, str | None]:
    patterns = [
        (r'bsd\s*[- ]?3\s*[- ]?clause|bsd-3-clause', 'BSD-3-Clause'),
        (r'bsd\s*[- ]?2\s*[- ]?clause|bsd-2-clause', 'BSD-2-Clause'),
        (r'\bmit\b(?:\s+license)?', 'MIT'),
        (r'apache\s*2\.0|apache-2\.0|apache license', 'Apache-2.0'),
        (r'gpl\s*v?3|gnu\s+general\s+public\s+license\s*v?3', 'GPL-3.0'),
        (r'gpl\s*v?2|gnu\s+general\s+public\s+license\s*v?2', 'GPL-2.0'),
        (r'lgpl\s*v?3|lesser\s+general\s+public\s+license\s*v?3', 'LGPL-3.0'),
        (r'mpl\s*2\.0|mozilla\s+public\s+license\s*2\.0', 'MPL-2.0'),
    ]

    low = readme_text.lower()
    for pat, spdx in patterns:
        m = re.search(pat, low, flags=re.IGNORECASE)
        if m:
            start = max(0, m.start() - 80)
            end = min(len(readme_text), m.end() + 80)
            evidence = readme_text[start:end].replace('\n', ' ').strip()
            return spdx, evidence

    return None, None


def _strip_html_tags(text: str) -> str:
    return re.sub(r'<[^>]+>', '', text or '').strip()


def parse_contributor_links_from_html(html_text: str) -> list[dict]:
    links: list[dict] = []
    seen_urls: set[str] = set()
    for href, label in re.findall(
        r'<a\s+[^>]*href=["\']([^"\']+)["\'][^>]*>([\s\S]*?)</a>',
        html_text,
        flags=re.IGNORECASE,
    ):
        if 'github.com/' not in href.lower():
            continue
        url = href.strip()
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)
        name = _strip_html_tags(label)
        if not name:
            name = url.rstrip('/').split('/')[-1]
        links.append({'name': name, 'github_url': url})
    return links


def extract_contributors_from_text(text: str) -> list[dict]:
    found: list[dict] = []
    seen_urls: set[str] = set()

    # Markdown links [name](https://github.com/user)
    for name, url in re.findall(
        r'\[([^\]]+)\]\((https?://(?:www\.)?github\.com/[A-Za-z0-9_.-]+(?:/[A-Za-z0-9_.-]+)?)\)',
        text,
        flags=re.IGNORECASE,
    ):
        u = url.strip()
        if u and u not in seen_urls:
            seen_urls.add(u)
            found.append({'name': name.strip(), 'github_url': u})

    # Bare GitHub profile URLs
    for url in re.findall(r'https?://(?:www\.)?github\.com/[A-Za-z0-9_.-]+(?:/[A-Za-z0-9_.-]+)?', text, flags=re.IGNORECASE):
        u = url.strip().rstrip(').,;')
        if u and u not in seen_urls:
            seen_urls.add(u)
            name = u.rstrip('/').split('/')[-1]
            found.append({'name': name, 'github_url': u})

    # @handles fallback
    for handle in re.findall(r'(?<![\w/])@([A-Za-z0-9-]{1,39})\b', text):
        u = f'https://github.com/{handle}'
        if u not in seen_urls:
            seen_urls.add(u)
            found.append({'name': handle, 'github_url': u})

    return found


def _normalize_contributor_list(items: list[dict]) -> list[dict]:
    merged: list[dict] = []
    seen_urls: set[str] = set()
    for item in items:
        if not isinstance(item, dict):
            continue
        url = str(item.get('github_url') or '').strip()
        name = str(item.get('name') or '').strip()
        if not url:
            continue
        if 'github.com/' not in url.lower():
            continue
        if url in seen_urls:
            continue
        if not name:
            name = url.rstrip('/').split('/')[-1]
        seen_urls.add(url)
        merged.append({'name': name, 'github_url': url})
    return merged


def _contributors_from_llm_json(raw: dict) -> list[dict]:
    value = raw.get('value')
    items: list[dict] = []

    if isinstance(value, list):
        for v in value:
            if isinstance(v, dict):
                items.append({'name': v.get('name'), 'github_url': v.get('github_url')})
            elif isinstance(v, str):
                items.extend(extract_contributors_from_text(v))
    elif isinstance(value, dict):
        if 'contributors' in value and isinstance(value['contributors'], list):
            for v in value['contributors']:
                if isinstance(v, dict):
                    items.append({'name': v.get('name'), 'github_url': v.get('github_url')})
                elif isinstance(v, str):
                    items.extend(extract_contributors_from_text(v))
        else:
            items.append({'name': value.get('name'), 'github_url': value.get('github_url')})
    elif isinstance(value, str):
        items.extend(extract_contributors_from_text(value))

    # Also parse evidence/source if provided
    for extra_field in ('evidence_quote', 'source_heading'):
        extra_val = raw.get(extra_field)
        if isinstance(extra_val, str):
            items.extend(extract_contributors_from_text(extra_val))

    return _normalize_contributor_list(items)


def extract_links_from_text(text: str) -> list[dict]:
    found: list[dict] = []
    seen: set[str] = set()

    # Markdown links [title](url)
    for title, url in re.findall(r'\[([^\]]+)\]\((https?://[^\s)]+)\)', text, flags=re.IGNORECASE):
        u = url.strip().rstrip(').,;')
        if u and u not in seen:
            seen.add(u)
            found.append({'title': title.strip(), 'url': u})

    # Bare URLs
    for url in re.findall(r'https?://[^\s<>()\]"\']+', text, flags=re.IGNORECASE):
        u = url.strip().rstrip(').,;')
        if u and u not in seen:
            seen.add(u)
            found.append({'title': None, 'url': u})

    return found


def _is_relevant_paper_link(url: str, title: str | None = None, context: str | None = None) -> bool:
    u = (url or '').lower()
    t = (title or '').lower()
    c = (context or '').lower()

    paper_domains = [
        'arxiv.org', 'doi.org', 'paperswithcode.com', 'openreview.net',
        'ieeexplore.ieee.org', 'dl.acm.org', 'springer.com', 'nature.com',
        'sciencedirect.com', 'biorxiv.org', 'medrxiv.org'
    ]
    if any(d in u for d in paper_domains):
        return True

    paper_terms = ['paper', 'publication', 'publications', 'cite', 'citation', 'preprint', 'manuscript']
    return any(term in u for term in paper_terms) or any(term in t for term in paper_terms) or any(term in c for term in paper_terms)


def check_link_status(url: str, timeout: int = 8) -> dict:
    try:
        head = requests.head(
            url,
            timeout=timeout,
            allow_redirects=True,
            headers={'User-Agent': 'maSMP-link-check/1.0'},
        )
        status = int(head.status_code)
        if status >= 400 or status == 405:
            get_resp = requests.get(
                url,
                timeout=timeout,
                allow_redirects=True,
                headers={'User-Agent': 'maSMP-link-check/1.0'},
                stream=True,
            )
            status = int(get_resp.status_code)
            final_url = str(get_resp.url)
            get_resp.close()
        else:
            final_url = str(head.url)

        return {
            'is_working': 200 <= status < 400,
            'status_code': status,
            'final_url': final_url,
            'error': None,
        }
    except Exception as exc:
        return {
            'is_working': False,
            'status_code': None,
            'final_url': None,
            'error': str(exc),
        }


def _normalize_link_list(items: list[dict], relevance_default: str = 'other') -> list[dict]:
    merged: list[dict] = []
    seen: set[str] = set()
    for item in items:
        if not isinstance(item, dict):
            continue
        url = str(item.get('url') or '').strip().rstrip(').,;')
        if not url or not re.match(r'^https?://', url, flags=re.IGNORECASE):
            continue
        if url in seen:
            continue
        seen.add(url)
        merged.append({
            'title': (str(item.get('title')).strip() if item.get('title') is not None else None),
            'url': url,
            'relevance': str(item.get('relevance') or relevance_default),
            'is_working': item.get('is_working'),
            'status_code': item.get('status_code'),
            'final_url': item.get('final_url'),
            'error': item.get('error'),
        })
    return merged


def _links_from_llm_json(raw: dict) -> list[dict]:
    value = raw.get('value')
    items: list[dict] = []

    if isinstance(value, list):
        for v in value:
            if isinstance(v, dict):
                items.append({
                    'title': v.get('title') or v.get('name'),
                    'url': v.get('url') or v.get('link') or v.get('href'),
                    'relevance': v.get('relevance') or 'other',
                })
            elif isinstance(v, str):
                for i in extract_links_from_text(v):
                    items.append({'title': i.get('title'), 'url': i.get('url'), 'relevance': 'other'})
    elif isinstance(value, dict):
        items.append({
            'title': value.get('title') or value.get('name'),
            'url': value.get('url') or value.get('link') or value.get('href'),
            'relevance': value.get('relevance') or 'other',
        })
    elif isinstance(value, str):
        for i in extract_links_from_text(value):
            items.append({'title': i.get('title'), 'url': i.get('url'), 'relevance': 'other'})

    return _normalize_link_list(items, relevance_default='other')


def _normalize_description_text(text: str | None) -> str | None:
    if not isinstance(text, str):
        return None
    compact = ' '.join(text.strip().split())
    if not compact:
        return None
    if len(compact) > 260:
        compact = compact[:257].rstrip() + '...'
    return compact


def build_property_prompt(repo_url: str, property_name: str, chunks: list[dict]) -> str:
    schema_hint = PROPERTY_SCHEMA_HINTS.get(property_name, 'string or null')
    if property_name == 'license':
        property_rule = 'For license, return a normalized SPDX label and an exact evidence quote if present. '
    elif property_name == 'contributors':
        property_rule = (
            'For contributors return STRICT JSON where value is a list of objects: '
            '[{"name": string, "github_url": string|null}]. Include ALL contributors found in provided chunks. '
        )
    elif property_name == 'links':
        property_rule = (
            'For links return STRICT JSON where value is a list of objects: '
            '[{"title": string|null, "url": string, "relevance": "paper"|"docs"|"other"}]. '
            'Focus on relevant paper/publication links and documentation links found in provided chunks only. '
        )
    elif property_name == 'description':
        property_rule = (
            'For description return brief factual lines about what the repository does, '
            'grounded only in the provided chunks. Avoid hype and guessing. '
        )
    else:
        property_rule = 'For non-license fields, return a concise value, plus a supporting quote when available. '

    context = []
    for c in chunks:
        context.append(
            f"[chunk_id={c['chunk_id']}; rank={c['rank']}; heading={c['heading']}]\n{c['content']}"
        )
    context_text = '\n\n-----\n\n'.join(context)

    return (
        f"Task: Extract property '{property_name}' from README evidence only. "
        f"{property_rule}"
        "No guessing. Return minified JSON only. "
        "If evidence is missing, set value=null, evidence_quote=null, source_heading=null, evidence_type='llm_assumption'. "
        "If evidence exists, evidence_quote should be an exact quote from the provided chunks whenever possible. "
        f"Schema: {{\"repository_url\": string, \"property\": string, \"value\": {schema_hint}, \"evidence_quote\": string|null, \"source_heading\": string|null, \"evidence_type\": \"readme_evidence\"|\"llm_assumption\", \"confidence\": number}}. "
        f"Repository URL: {repo_url}\n\n"
        f"Retrieved README chunks:\n{context_text}"
    )


def validate_property_output(raw: dict, property_name: str, chunks: list[dict], repo_url: str) -> dict:
    merged_text = '\n'.join([c['content'] for c in chunks]).lower()
    headings = [str(c.get('heading') or '').strip().lower() for c in chunks if c.get('heading')]
    property_terms = PROPERTY_QUERIES.get(property_name, [])
    property_support = any(term.lower() in merged_text for term in property_terms)

    out = {
        'repository_url': repo_url,
        'property': property_name,
        'value': None,
        'evidence_quote': None,
        'source_heading': None,
        'evidence_type': 'llm_assumption',
        'confidence': 0.0,
    }
    if not isinstance(raw, dict):
        return out

    out['repository_url'] = str(raw.get('repository_url') or repo_url).strip()
    out['property'] = str(raw.get('property') or property_name).strip()
    out['value'] = raw.get('value')
    out['confidence'] = float(raw.get('confidence') or 0.0)

    raw_heading = raw.get('source_heading')
    heading_support = isinstance(raw_heading, str) and raw_heading.strip().lower() in headings
    if heading_support:
        out['source_heading'] = str(raw_heading).strip()

    quote = raw.get('evidence_quote')
    quote_text = quote.strip() if isinstance(quote, str) and quote.strip() else None
    has_exact_quote = bool(quote_text and quote_text.lower() in merged_text)

    if property_name == 'license':
        if has_exact_quote and out['value'] is not None:
            out['evidence_quote'] = quote_text
            out['evidence_type'] = 'readme_evidence'
            out['confidence'] = max(out['confidence'], 0.6)
        else:
            out['value'] = None
            out['evidence_quote'] = None
            out['source_heading'] = None
            out['evidence_type'] = 'llm_assumption'
            out['confidence'] = min(out['confidence'], 0.3)
        return out

    if property_name == 'contributors':
        contribs = _normalize_contributor_list(out['value'] if isinstance(out['value'], list) else [])
        out['value'] = contribs if contribs else None
        if out['value'] is None:
            out['evidence_type'] = 'llm_assumption'
            out['confidence'] = min(out['confidence'], 0.3)
            out['evidence_quote'] = None
            out['source_heading'] = None
        else:
            out['evidence_type'] = 'readme_evidence'
            out['confidence'] = max(out['confidence'], 0.7)
            if not quote_text:
                out['evidence_quote'] = None
        return out

    if property_name == 'links':
        links = _normalize_link_list(out['value'] if isinstance(out['value'], list) else [])
        out['value'] = links if links else None
        if out['value'] is None:
            out['evidence_type'] = 'llm_assumption'
            out['confidence'] = min(out['confidence'], 0.3)
            out['evidence_quote'] = None
            out['source_heading'] = None
        else:
            out['evidence_type'] = 'readme_evidence'
            out['confidence'] = max(out['confidence'], 0.7)
            if not quote_text:
                out['evidence_quote'] = None
        return out

    if property_name == 'description':
        out['value'] = _normalize_description_text(out['value'])
        if out['value'] is None:
            out['evidence_type'] = 'llm_assumption'
            out['confidence'] = min(out['confidence'], 0.3)
            out['evidence_quote'] = None
            out['source_heading'] = None
        else:
            out['evidence_type'] = 'readme_evidence' if (has_exact_quote or heading_support or property_support) else 'llm_assumption'
            out['confidence'] = max(out['confidence'], 0.65 if out['evidence_type'] == 'readme_evidence' else 0.4)
            if out['evidence_type'] == 'llm_assumption':
                out['evidence_quote'] = None
                out['source_heading'] = None
            elif not quote_text:
                out['evidence_quote'] = None
        return out

    if out['value'] in (None, '', []):
        out['evidence_quote'] = None
        out['source_heading'] = None
        out['evidence_type'] = 'llm_assumption'
        out['confidence'] = min(out['confidence'], 0.3)
        return out

    if has_exact_quote:
        out['evidence_quote'] = quote_text
        out['evidence_type'] = 'readme_evidence'
        out['confidence'] = max(out['confidence'], 0.65)
    elif heading_support or property_support:
        out['evidence_quote'] = quote_text
        out['evidence_type'] = 'readme_evidence'
        out['confidence'] = max(out['confidence'], 0.5)
    else:
        out['evidence_quote'] = None
        out['source_heading'] = None
        out['evidence_type'] = 'llm_assumption'
        out['confidence'] = min(out['confidence'], 0.4)

    return out


def extract_property_with_retrieval(repo_url: str, property_name: str, model: str, ollama_url: str, top_k: int = 5) -> dict:
    if property_name == 'license':
        k, alpha = 4, 0.8
    elif property_name == 'contributors':
        k, alpha = max(top_k, 20), 0.65
    elif property_name == 'links':
        k, alpha = max(top_k, 12), 0.70
    elif property_name == 'description':
        k, alpha = max(top_k, 6), 0.75
    else:
        k, alpha = top_k, 0.75

    picked = retrieve_top_chunks(retrieval_index, property_name=property_name, top_k=k, alpha=alpha)
    prompt = build_property_prompt(repo_url=repo_url, property_name=property_name, chunks=picked)
    model_output = run_ollama(prompt=prompt, model=model, ollama_url=ollama_url, timeout=420)

    if property_name == 'contributors':
        seed_items: list[dict] = []
        for c in picked:
            seed_items.extend(extract_contributors_from_text(f"{c.get('heading','')}\n{c.get('content','')}"))
        seed_items = _normalize_contributor_list(seed_items)

        llm_items: list[dict] = []
        try:
            raw = extract_json(model_output)
            llm_items = _contributors_from_llm_json(raw)
        except ValueError:
            try:
                html_snippet = extract_html(model_output)
                llm_items = parse_contributor_links_from_html(html_snippet)
            except ValueError:
                llm_items = extract_contributors_from_text(model_output)

        merged_items = _normalize_contributor_list(seed_items + llm_items)
        raw = {
            'repository_url': repo_url,
            'property': 'contributors',
            'value': merged_items if merged_items else None,
            'evidence_quote': None,
            'source_heading': picked[0]['heading'] if picked else None,
            'evidence_type': 'readme_evidence' if merged_items else 'llm_assumption',
            'confidence': 0.75 if merged_items else 0.2,
        }
    elif property_name == 'links':
        seed_items: list[dict] = []
        for c in picked:
            combined = f"{c.get('heading','')}\n{c.get('content','')}"
            for item in extract_links_from_text(combined):
                relevance = 'paper' if _is_relevant_paper_link(item.get('url', ''), item.get('title'), combined) else 'other'
                if relevance == 'paper' or 'doc' in combined.lower() or 'readthedocs' in item.get('url', '').lower():
                    seed_items.append({
                        'title': item.get('title'),
                        'url': item.get('url'),
                        'relevance': 'paper' if relevance == 'paper' else 'docs',
                    })

        llm_items: list[dict] = []
        try:
            raw = extract_json(model_output)
            llm_items = _links_from_llm_json(raw)
        except ValueError:
            for item in extract_links_from_text(model_output):
                llm_items.append({
                    'title': item.get('title'),
                    'url': item.get('url'),
                    'relevance': 'paper' if _is_relevant_paper_link(item.get('url', ''), item.get('title'), model_output) else 'other',
                })

        merged_items = _normalize_link_list(seed_items + llm_items)

        checked_items: list[dict] = []
        for item in merged_items[:30]:
            status = check_link_status(item['url'])
            checked_items.append({
                **item,
                'is_working': status['is_working'],
                'status_code': status['status_code'],
                'final_url': status['final_url'],
                'error': status['error'],
            })

        raw = {
            'repository_url': repo_url,
            'property': 'links',
            'value': checked_items if checked_items else None,
            'evidence_quote': None,
            'source_heading': picked[0]['heading'] if picked else None,
            'evidence_type': 'readme_evidence' if checked_items else 'llm_assumption',
            'confidence': 0.75 if checked_items else 0.2,
        }
    elif property_name == 'description':
        try:
            raw = extract_json(model_output)
        except ValueError:
            raw = {
                'repository_url': repo_url,
                'property': 'description',
                'value': _normalize_description_text(model_output),
                'evidence_quote': None,
                'source_heading': picked[0]['heading'] if picked else None,
                'evidence_type': 'readme_evidence' if model_output else 'llm_assumption',
                'confidence': 0.45 if model_output else 0.1,
            }
    else:
        raw = extract_json(model_output)

    validated = validate_property_output(raw=raw, property_name=property_name, chunks=picked, repo_url=repo_url)
    validated['retrieved_chunk_ids'] = [c['chunk_id'] for c in picked]
    validated['retrieved_headings'] = [c['heading'] for c in picked]
    validated['retrieved_scores'] = [c.get('score') for c in picked]
    return validated

In [171]:
# Description is now integrated directly in the main extraction functions above.
# This cell is intentionally kept as a no-op to avoid breaking notebook order.

In [172]:
# run property extraction with retrieval
properties_to_extract = ['license', 'installation', 'contact', 'contributors', 'links', 'description']
results = {}

for property_name in properties_to_extract:
    try:
        results[property_name] = extract_property_with_retrieval(
            repo_url=repo_url_preview,
            property_name=property_name,
            model='qwen2.5:7b',
            ollama_url='http://localhost:11435',
            top_k=5,
        )
    except Exception as exc:
        results[property_name] = {'property': property_name, 'error': str(exc)}

license_rule, license_evidence = extract_license_from_readme(readme_text_preview)
if license_rule and isinstance(results.get('license'), dict):
    results['license']['value'] = license_rule
    results['license']['evidence_quote'] = license_evidence
    results['license']['source_heading'] = 'regex_rule'
    results['license']['evidence_type'] = 'readme_rule'
    results['license']['confidence'] = max(float(results['license'].get('confidence') or 0.0), 0.95)

print(json.dumps(results, indent=2, ensure_ascii=False))

{
  "license": {
    "repository_url": "https://github.com/CERN/TIGRE",
    "property": "license",
    "value": "BSD-3-Clause",
    "evidence_quote": "It is released under the BSD License, meaning you can use and modify the software freely.",
    "source_heading": "Licensing",
    "evidence_type": "readme_evidence",
    "confidence": 1.0,
    "retrieved_chunk_ids": [
      10,
      11,
      18,
      1
    ],
    "retrieved_headings": [
      "Licensing",
      "Licensing",
      "Contributors",
      "TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox"
    ],
    "retrieved_scores": [
      0.6843735575675964,
      0.5801926255226135,
      0.5359349250793457,
      0.5181072354316711
    ]
  },
  "installation": {
    "repository_url": "https://github.com/CERN/TIGRE",
    "property": "installation",
    "value": null,
    "evidence_quote": null,
    "source_heading": null,
    "evidence_type": "llm_assumption",
    "confidence": 0.3,
    "retrieved_chunk_ids": [
      4

In [173]:
contact_result = extract_property_with_retrieval(
    repo_url=repo_url_preview,
    property_name='contact',
    model='qwen2.5:7b',
    ollama_url='http://localhost:11435',
    top_k=5,
 )

print(json.dumps(contact_result, indent=2, ensure_ascii=False))

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "contact",
  "value": "[tigre.toolbox@gmail.com](mailto:tigre.toolbox@gmail.com) or [ander.biguri@gmail.com](mailto:ander.biguri@gmail.com)",
  "evidence_quote": "Contact the authors directly at:\n\n[tigre.toolbox@gmail.com](mailto:tigre.toolbox@gmail.com) or [ander.biguri@gmail.com](mailto:ander.biguri@gmail.com)",
  "source_heading": "Contact",
  "evidence_type": "readme_evidence",
  "confidence": 1.0,
  "retrieved_chunk_ids": [
    9,
    17,
    12,
    13,
    16
  ],
  "retrieved_headings": [
    "Contact",
    "Contributors",
    "Contributors",
    "Contributors",
    "Contributors"
  ],
  "retrieved_scores": [
    0.775506854057312,
    0.5279214382171631,
    0.5151097774505615,
    0.514689028263092,
    0.5098159909248352
  ]
}


In [174]:
# Deterministic override for license
license_rule, license_evidence = extract_license_from_readme(readme_text_preview)

if 'license' not in results:
    results['license'] = {
        'repository_url': repo_url_preview,
        'property': 'license',
        'value': None,
        'evidence_quote': None,
        'source_heading': None,
        'evidence_type': 'llm_assumption',
        'confidence': 0.0,
    }

if license_rule:
    results['license']['value'] = license_rule
    results['license']['evidence_quote'] = license_evidence
    results['license']['source_heading'] = 'regex_rule'
    results['license']['evidence_type'] = 'readme_rule'
    results['license']['confidence'] = max(float(results['license'].get('confidence') or 0.0), 0.95)

print(json.dumps(results['license'], indent=2, ensure_ascii=False))

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "license",
  "value": "BSD-3-Clause",
  "evidence_quote": "It is released under the BSD License, meaning you can use and modify the software freely.",
  "source_heading": "Licensing",
  "evidence_type": "readme_evidence",
  "confidence": 1.0,
  "retrieved_chunk_ids": [
    10,
    11,
    18,
    1
  ],
  "retrieved_headings": [
    "Licensing",
    "Licensing",
    "Contributors",
    "TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox"
  ],
  "retrieved_scores": [
    0.6843735575675964,
    0.5801926255226135,
    0.5359349250793457,
    0.5181072354316711
  ]
}


In [175]:
# Demo: extract relevant links (paper/docs) and verify accessibility
links_result = extract_property_with_retrieval(
    repo_url=repo_url_preview,
    property_name='links',
    model='qwen2.5:7b',
    ollama_url='http://localhost:11435',
    top_k=12,
 )

print(json.dumps(links_result, indent=2, ensure_ascii=False))

if isinstance(links_result.get('value'), list):
    working = [x for x in links_result['value'] if x.get('is_working')]
    failing = [x for x in links_result['value'] if not x.get('is_working')]
    print(f"Working links: {len(working)} | Failing links: {len(failing)}")

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "links",
  "value": [
    {
      "title": null,
      "url": "https://doi.org/10.1016/j.jpdc.2020.07.004",
      "relevance": "paper",
      "is_working": true,
      "status_code": 200,
      "final_url": "https://linkinghub.elsevier.com/retrieve/pii/S0743731520303336",
      "error": null
    },
    {
      "title": null,
      "url": "https://arxiv.org/abs/1905.03748",
      "relevance": "paper",
      "is_working": true,
      "status_code": 200,
      "final_url": "https://arxiv.org/abs/1905.03748",
      "error": null
    },
    {
      "title": null,
      "url": "https://github.com/CERN/TIGRE",
      "relevance": "paper",
      "is_working": true,
      "status_code": 200,
      "final_url": "https://github.com/CERN/TIGRE",
      "error": null
    },
    {
      "title": null,
      "url": "http://iopscience.iop.org/article/10.1088/2057-1976/2/5/055010",
      "relevance": "paper",
      "is_working": true,
 

This has the dedicated llm_model (qwen2.5:7b), to extract the information from the README of a particular repo.
- Properties like (license, link, installation, contact, contributors)